# 01 · Spectra, bandpasses, and count rates

**Roman GRS disperser tutorials.** This is the foundations notebook: before we disperse anything, we need to turn an *astronomical source* into the quantity the disperser actually consumes — a **count-rate spectrum**, i.e. detected electrons per second as a function of wavelength.

By the end you will be able to:

1. load a Roman WFI **bandpass** and a **spectral template** from the data bundled with `roman_disperser`;
2. **normalize** a template to a given AB magnitude;
3. sample it onto a **wavelength grid** — and keep the *microns vs Ångströms* bookkeeping straight;
4. apply a grism **sensitivity curve** to convert flux into a **count rate**.

We finish by packaging steps 1–4 into a one-line helper (`tutorial_helpers.template_to_counts`) that every later notebook reuses.

> **Prerequisites.** A working tutorials environment with the reference data hydrated (`pixi run hydrate`, or `roman-disperser-hydrate` for users). See `docs/SETUP.md`. This notebook only needs the `optical_model`, `sensitivities`, and `synphot` assets — it does **not** disperse, so it runs in well under a minute on a laptop CPU.

## 0 · Setup

In [ ]:
from importlib.metadata import version

import numpy as np
import matplotlib.pyplot as plt

import astropy.units as u
import synphot as syn

from roman_disperser import refdata, paths
from roman_disperser.elements import GRISM

# tutorial_helpers.py lives alongside the notebooks; the helper we build up by
# hand below is also packaged there so notebooks 02+ can import it directly.
import tutorial_helpers as th

print("roman_disperser:", version("roman_disperser"))
print("reference data :", paths.data_dir())

### The dispersing element

WFI carries two dispersing elements: the **grism** (three spectral orders, 0.9–2.0 µm — the GRS workhorse) and the **prism** (a single order over 0.75–1.85 µm at lower spectral resolution). Everything element-specific — the orders, the band edges, the STPSF filter per order, which sensitivity files to use — is bundled into a `DispersingElement` constant: `roman_disperser.elements.GRISM` or `PRISM`. Passing `element=` to the disperser's functions keeps those choices consistent, instead of you juggling them by hand.

Notebooks 01–08 use the grism throughout; [notebook 09](09_prism.ipynb) swaps in the prism and shows that essentially nothing else changes.

## 1 · The bandpass

We anchor source magnitudes in the Roman WFI **F158** wide filter (a convenient reference across the grism's ~0.9–2.0 µm range); `roman_disperser` bundles it. `synphot` gives the usual summary numbers — pivot wavelength, FWHM, throughput.

In [ ]:
f158 = refdata.get_f158_band()

print(f"F158 pivot wavelength : {f158.pivot():.1f}")
print(f"F158 FWHM             : {f158.fwhm():.1f}")
print(f"F158 wave range       : {f158.waverange[0]:.0f} – {f158.waverange[1]:.0f}")
print(f"throughput at 1.58 um : {f158(1.58*u.micron):.3f}")

In [ ]:
wl_plot = np.linspace(1.0, 2.1, 400) * u.micron
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(wl_plot.to(u.micron), f158(wl_plot), color="C0")
ax.axvline(f158.pivot().to(u.micron).value, ls="--", color="0.5",
           label=f"pivot = {f158.pivot().to(u.micron):.3f}")
ax.set(xlabel="wavelength [µm]", ylabel="throughput", title="Roman WFI F158 bandpass")
ax.legend()
fig.tight_layout()

## 2 · Spectral templates and normalization

`roman_disperser` bundles a small atlas of spectral templates (synphot `SourceSpectrum` objects):

| name | kind | rest-frame coverage |
|------|------|---------------------|
| `g0v`, `bz77_bz_24` | stellar (Bruzual 1977) | ~240–34000 Å (covers F158) |
| `kc96_elliptical`, `kc96_starb1` | galaxy (Kinney–Calzetti) | ~1235–9945 Å (UV–optical only) |

Set the flux by normalizing to an AB magnitude in a band: `SourceSpectrum.normalize(mag * u.ABmag, band=...)`. Note the galaxy templates are **rest-frame** and reach only ~9945 Å, so they must be **redshifted** into the grism band before use (§6).

We'll start with a star — a **G0V** template normalized to AB = 18 in F158.

In [ ]:
MAG = 18.0
star = refdata.get_template("g0v").normalize(MAG * u.ABmag, band=f158)
print(f"normalized a G0V template to AB = {MAG} in F158")

## 3 · The wavelength grid — microns vs Ångströms

A frequent source of silent bugs, so we keep **both** representations side by side:

- **synphot** and the **sensitivity FITS files** speak **Ångströms**;
- the **optical model / PSF / disperser** (notebooks 02+) speak **microns**.

We sample on the grid the production catalog uses — **2 Å** spacing across 0.9–2.0 µm (~5500 points).

In [ ]:
# helper returns (microns, angstroms, dlam_angstrom); see tutorial_helpers.py.
# The band edges come from the element; th.wavelength_grid(GRISM) is the shorthand.
wl_um, wl_a, dlam_a = th.grism_wavelength_grid(lam_min_um=GRISM.lam_min,
                                               lam_max_um=GRISM.lam_max,
                                               dlam_angstrom=2.0)
print(f"{wl_um.size} samples, Δλ = {dlam_a} Å")
print(f"first/last (µm): {wl_um[0]:.3f} … {wl_um[-1]:.3f}")
print(f"first/last (Å) : {wl_a[0]:.0f} … {wl_a[-1]:.0f}")

In [ ]:
# Evaluate the normalized template ON the grid, as f_lambda (FLAM = erg/s/cm²/Å).
# synphot wants wavelengths WITH units — pass Ångströms.
flam = star(wl_a * u.AA, flux_unit=syn.units.FLAM).value

fig, ax = plt.subplots(figsize=(7, 3))
ax.semilogy(wl_um, flam, color="C1")
ax.set(xlabel="wavelength [µm]", ylabel=r"$f_\lambda$ [erg s$^{-1}$ cm$^{-2}$ Å$^{-1}$]",
       title=f"G0V template, AB={MAG} in F158")
fig.tight_layout()

## 4 · Grism sensitivity curves

The **sensitivity** curve folds together everything between flux at the telescope and electrons on the detector for a given **SCA** and **order**: collecting area, grism efficiency into that order, throughput, QE, exposure-time bookkeeping. One curve per detector (1–18) and order — the grism's orders 0, 1, 2 in `data/sensitivities/`, the prism's single order 1 in `data/sensitivities_prism/` (each directory indexed by its `sensitivity_map.yaml`; the element's `sensitivities_subdir` field picks the right one).

**Units.** It is defined so that `FLAM × sensitivity × Δλ(Å)` yields a count rate in e⁻/s — i.e. it carries (e⁻/s) per (erg s⁻¹ cm⁻² Å⁻¹ · Å). We load **SCA 5, order 1** and interpolate onto our grid (zero outside the tabulated range).

In [ ]:
SCA, ORDER = 5, "1"

# load_sensitivity reads the element's sensitivity_map.yaml, opens the right
# FITS, and np.interp's it onto our Ångström grid (zero outside the tabulated
# range). element= defaults to the grism; we spell it out once for clarity.
sens = th.load_sensitivity(SCA, ORDER, wl_a, element=GRISM)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(wl_um, sens, color="C2")
ax.set(xlabel="wavelength [µm]", ylabel="sensitivity  [counts/s per FLAM·Å]",
       title=f"Grism sensitivity — SCA{SCA}, order {ORDER}")
fig.tight_layout()

## 5 · From flux to counts

The count rate in each wavelength bin is

$$ C_i \;=\; f_\lambda(\lambda_i)\; \times\; S(\lambda_i)\; \times\; \Delta\lambda $$

with $f_\lambda$ in FLAM, the sensitivity $S$ in counts/s per (FLAM·Å), and the bin width $\Delta\lambda$ in **Ångströms** — so the units collapse to **electrons per second** per bin. That $\Delta\lambda$ factor is easy to forget and silently rescales everything, so we keep it explicit.

In [ ]:
counts = flam * sens * dlam_a   # electrons / s, per wavelength bin

print(f"total order-1 count rate : {counts.sum():8.1f} e-/s")
print(f"peak per-bin count rate  : {counts.max():8.3f} e-/s")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(wl_um, counts, color="C3")
ax.set(xlabel="wavelength [µm]", ylabel="count rate [e⁻/s per bin]",
       title=f"G0V, AB={MAG} → counts (SCA{SCA}, order {ORDER})")
fig.tight_layout()

## 6 · Packaging it — and a galaxy

Steps 2–5 are the boilerplate every later notebook repeats, so they're packaged as `tutorial_helpers.template_to_counts(...)`; the code above *is* its body. For a galaxy we pass `redshift=` (see §2) so the rest-frame template lands in the grism band — here an elliptical at z = 1.5, whose 4000 Å break falls in range.

In [ ]:
wl_star, c_star = th.template_to_counts("g0v", 18.0, sca=SCA, order=ORDER)
wl_gal,  c_gal  = th.template_to_counts("kc96_elliptical", 21.0, sca=SCA, order=ORDER,
                                        redshift=1.5)

# Star and galaxy differ by ~100x in total counts, so give each its own panel.
fig, (a0, a1) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
a0.plot(wl_star, c_star, color="C0")
a0.set(ylabel="e⁻/s per bin", title="G0V star, AB=18")
a1.plot(wl_gal, c_gal, color="C4")
a1.set(xlabel="wavelength [µm]", ylabel="e⁻/s per bin",
       title="elliptical galaxy, AB=21, z=1.5")
fig.tight_layout()

print(f"star total: {c_star.sum():.1f} e-/s    galaxy total: {c_gal.sum():.1f} e-/s")

## Recap

- A source becomes a **count-rate spectrum** via: template → normalize (in a band) → sample as FLAM → × sensitivity × Δλ.
- Keep wavelengths in **µm for the disperser**, **Å for synphot/sensitivities**.
- Galaxies must be **redshifted** into the grism band before normalizing.
- The **dispersing element** (`elements.GRISM`/`PRISM`) carries the band, the orders, and which sensitivity directory to read — pass it rather than hand-picking those.
- `tutorial_helpers.template_to_counts(name, mag, sca, order, redshift=..., element=...)` wraps the whole chain and returns `(wl_um, counts)`.

**Next — [02 · Disperse a star on one SCA](02_disperse_a_star.ipynb).** We take this count-rate spectrum, attach it to a point source at a fixed detector position, and run it through the optical model + PSF to produce a dispersed first-order spectrum on the SCA — introducing the JAX warm-up step along the way.